In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing as ES 
from sklearn.model_selection import TimeSeriesSplit
import numpy as np

In [2]:
df = pd.read_csv('Auto_sales.csv', sep=';', \
    decimal=',', parse_dates=['Date'], \
    index_col='Date')
df = df.dropna()

In [3]:
n_splits = 200
tscv = TimeSeriesSplit(n_splits=n_splits,test_size=1)

In [4]:
mse_ses = np.zeros(n_splits)
mse_des = np.zeros(n_splits)
mse_tes = np.zeros(n_splits)

In [5]:
def calc_mse(model,data,test_idx):
    y=data[test_idx[0]]

    y_hat = model.predict(test_idx[0])

    mse = (y_hat-y)**2

    return mse


In [7]:
k = 0 

for train_index, test_index in tscv.split(df.Auto_sales):
    train_data = df.Auto_sales[train_index]
    test_data = df.Auto_sales[test_index]

    ses = ES(train_data).fit()
    mse_ses[k] = calc_mse(ses,df.Auto_sales,test_index)

    des = ES(train_data,trend='add').fit()
    mse_des = calc_mse(des,df.Auto_sales,test_index)

    tes = ES(train_data,trend='add', seasonal='mul', seasonal_periods=12).fit()
    mse_tes = calc_mse(tes,df.Auto_sales,test_index)

    k+=1

C:\Users\mrosk\AppData\Local\Temp\ipykernel_15420\1611724019.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  train_data = df.Auto_sales[train_index]
C:\Users\mrosk\AppData\Local\Temp\ipykernel_15420\1611724019.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  test_data = df.Auto_sales[test_index]
c:\Users\mrosk\.conda\envs\ml311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
C:\Users\mrosk\AppData\Local\Temp\ipykernel_15420\1654069091.py:2: FutureWarning: Series.__getitem__ treat

In [8]:
print("SES MSE:" , np.sqrt(mse_ses.mean()), '\n', \
      "DES MSE:", np.sqrt(mse_des.mean()) , '\n', \
      "TES MSE:", np.sqrt(mse_tes.mean()))

SES MSE: 76.2932525913538 
 DES MSE: 33.18002455516796 
 TES MSE: 42.91630009207984
